# Notebook 03: Bài toán Học máy — Huấn luyện trên 2025 (Q1–Q4) & Dự đoán/Kiểm thử trên 2026 (Q1) (Q6) + Tổng kết Đồ án
**Đề tài:** Phân tích Sự cố Bất lợi Thuốc Thú y (OpenFDA Animal & Veterinary Adverse Events) giai đoạn 2025 và Dự báo cho năm 2026 (Q1)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Nhóm:** Global Ban  
**Chủ sở hữu câu hỏi ML (Q6):** Đinh Công Khang (24120332)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.inspection import permutation_importance

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import log_row_count_change
from src.visualization import set_project_plot_style
set_project_plot_style()

---
## 1. Định nghĩa Bài toán Dự đoán & Chống Rò rỉ Dữ liệu (ML Requirement 1)

### 1.1. Câu hỏi Q6 & Người dùng mục tiêu (Target User & Decision)
- **Câu hỏi Q6:** Khi một báo cáo sự cố bất lợi mới được gửi đến hệ thống trong **năm 2026 (Q1)**, mô hình học máy được huấn luyện trên dữ liệu lịch sử **2025 (Q1–Q4)** có thể dự đoán chính xác khả năng sự cố đó thuộc nhóm **Nghiêm trọng (`serious_ae = True`)** dựa trên thông tin ban đầu của vật nuôi, thuốc và người báo cáo ở mức độ nào? Ngoài ra, tổng lượng báo cáo nghiêm trọng theo tuần/tháng trong Q1/2026 có khớp với xu hướng dự báo từ năm 2025 hay không?
- **Người dùng mục tiêu (User):** Chuyên viên thẩm định an toàn thuốc tại FDA CVM hoặc hệ thống cảnh báo sớm tại bệnh viện thú y — cần tự động phân loại ưu tiên (triage) các báo cáo có nguy cơ nghiêm trọng cao ngay khi tiếp nhận hồ sơ.

### 1.2. Đặc trưng đầu vào hợp lệ vs. Các cột phải loại bỏ để tránh Rò rỉ Dữ liệu (Data Leakage Prevention)
- **Đặc trưng hợp lệ tại thời điểm dự đoán (Available at prediction time):**
  - Đặc điểm động vật: `species`, `gender`, `reproductive_status`, `age_years`, `weight_kg`, `number_of_animals_affected`.
  - Đặc điểm thuốc & phơi nhiễm: `num_drugs`, `route`, `dosage_form`, `atc_vet_code`, `used_according_to_label`, nhóm hoạt chất chính (`top_active_ingredients`).
  - Thông tin ngữ cảnh tiếp nhận: `primary_reporter`, `type_of_information`, `receive_month` / `quarter`, `reporting_delay_days`.
- **Các cột BẮT BUỘC LOẠI BỎ để tránh Data Leakage:**
  1. `outcome` (`medical_status` như `Died`, `Euthanized`, `Recovered`): Kết quả lâm sàng (Clinical Outcome) xảy ra **sau** sự cố và là thành phần trực tiếp định nghĩa một ca có phải là `serious_ae` hay không.
  2. `treated_for_ae`: Việc có phải can thiệp điều trị cho phản ứng phụ hay không diễn ra sau khi phản ứng đã bộc lộ mức độ nghiêm trọng.
  3. `duration` (thời gian kéo dài của phản ứng): Chỉ biết sau khi sự kiện kết thúc.
  4. Mã định danh (`unique_aer_id_number`, `report_id`) và các nhãn hậu kiểm nội bộ.

---
## 2. Thiết kế Chia Tập dữ liệu theo Thời gian (Time-based Split — ML Requirement 2)

Vì dữ liệu có tính thứ tự thời gian rõ rệt từ đầu năm 2025 đến nửa đầu năm 2026, việc chia ngẫu nhiên (`random train_test_split`) sẽ làm rò rỉ thông tin tương lai (data leakage across time). Do đó, nhóm áp dụng **Time-based Split** nghiêm ngặt:
- **Tập Huấn luyện (Training & Temporal Validation Set):** Toàn bộ báo cáo từ **2025 Q1 đến 2025 Q4** (4 quý).
  - Trong nội bộ 2025 Q1–Q4, sử dụng `TimeSeriesSplit` trên 4 quý (tối đa 3 fold mở rộng: Q1→Q2; Q1–Q2→Q3; Q1–Q3→Q4) để tinh chỉnh siêu tham số (hyperparameter tuning). Kết quả CV cần được diễn giải thận trọng do số điểm thời gian ít.
- **Tập Kiểm thử Thực tế (Out-of-Time Test Set):** Toàn bộ báo cáo của **năm 2026 (Q1)**.
  - Tập 2026 Q1 được giữ kín hoàn toàn trong suốt quá trình thiết kế đặc trưng và tinh chỉnh mô hình, chỉ dùng để **đánh giá đúng 1 lần cuối cùng**.

In [ ]:
# TODO [Giai đoạn 4]: Tải bảng đặc trưng từ ../data/processed/, loại bỏ các cột leakage
# và tách X_train, y_train (2025 Q1-Q4) vs. X_test_2026, y_test_2026 (2026 Q1)


---
## 3. Xây dựng Pipeline, Baseline, Huấn luyện & Tinh chỉnh Siêu tham số (ML Requirements 3 & 4)

### 3.1. Kiến trúc `scikit-learn Pipeline`
- Toàn bộ các bước xử lý khuyết (`SimpleImputer`), chuẩn hóa thang đo (`StandardScaler`) và mã hóa biến phân loại (`OneHotEncoder(handle_unknown='ignore')`) đều nằm bên trong `ColumnTransformer` và `Pipeline` để tránh rò rỉ thống kê từ tập Validation/Test2026 vào tập Train.

### 3.2. Các mô hình so sánh
1. **Trivial Baseline:** `DummyClassifier(strategy='most_frequent')` (và baseline theo tỷ lệ lớp năm 2025).
2. **Mô hình tuyến tính:** `LogisticRegression(max_iter=1000, class_weight='balanced')`.
3. **Mô hình phi tuyến / Ensemble:** `HistGradientBoostingClassifier` và/hoặc `RandomForestClassifier`.

### 3.3. Thước đo đánh giá (Evaluation Metrics Justification)
- Chọn **PR-AUC (Average Precision)** và **ROC-AUC** làm thước đo chính khi tinh chỉnh với `TimeSeriesSplit`, kèm theo **F1-score / Recall** ở ngưỡng tối ưu.
- **Lý do:** Trong bài toán cảnh giác dược, việc bỏ sót một sự cố nghiêm trọng (False Negative) nguy hiểm hơn việc cảnh báo nhầm (False Positive), đồng thời tỷ lệ lớp có thể biến động giữa các năm nên `Accuracy` đơn thuần không phản ánh đúng chất lượng phân tách xác suất.

In [ ]:
# TODO [Giai đoạn 4]: Khởi tạo ColumnTransformer + Pipeline, chạy TimeSeriesSplit CV trên tập 2025 Q1-Q4
# và đánh giá 1 lần trên tập Test 2026 (Q1)


---
## 4. Phân tích Lỗi trên Tập Kiểm thử 2026 Q1 (Error Analysis — ML Requirement 5)
- Đánh giá độ chính xác của mô hình trên quý đầu của năm 2026 (`2026 Q1`) để kiểm tra hiện tượng **Temporal Drift / Concept Drift**.
- Phân tích tỷ lệ lỗi (False Positives & False Negatives) theo từng phân nhóm: Loài (`Dog`, `Cat`, `Cattle`, `Horse`), Đối tượng báo cáo (`Veterinarian` vs. `Animal Owner`), và các khoảng giá trị tuổi/cân nặng.
- Trích xuất một số dòng báo cáo thực tế năm 2026 mà mô hình dự đoán sai nặng nhất để giải thích nguyên nhân.

In [ ]:
# TODO [Giai đoạn 4]: Vẽ Confusion Matrix, đường cong ROC & PR trên tập 2026 Q1,
# và bảng phân rã lỗi theo subgroup + hiển thị các ca điển hình


---
## 5. Giải thích Mô hình & Trả lời Câu hỏi Nghiên cứu Q6 (Model Interpretation — ML Requirement 6)
- Sử dụng **Permutation Importance** trên tập kiểm thử 2026 (Q1) và hệ số mô hình để xác định những đặc trưng nào đóng góp lớn nhất vào dự đoán `serious_ae`.
- Kết luận rõ ràng bằng số liệu thực nghiệm và ý nghĩa thực tiễn.

In [ ]:
# TODO [Giai đoạn 4]: Tính và vẽ biểu đồ Permutation Importance trên tập 2026 Q1


---
## 6. Tổng kết Đồ án & Phản ánh Cá nhân (Summary and Reflection — Mục 3.5)

### 6.1. Các phát hiện quan trọng nhất (Key Findings — 3 đến 5 insights)
- *(Tổng hợp kết quả nổi bật nhất từ Q1–Q6 sau khi chạy thực nghiệm, nhấn mạnh phát hiện bất ngờ nhất)*

### 6.2. Hạn chế của Dữ liệu, Phương pháp và Phạm vi (Limitations)
- Hạn chế của dữ liệu báo cáo tự nguyện openFDA (thiếu mẫu số tổng đàn, thiên lệch báo cáo, dữ liệu chưa xác thực nhân quả).
- Hạn chế do rút gọn dữ liệu huấn luyện còn 4 quý năm 2025: ít điểm thời gian, CV kém ổn định và không đủ cơ sở khẳng định chu kỳ mùa vụ dài hạn.
- Hạn chế về thời gian của năm 2026 (mới có Q1 nên chưa phản ánh trọn vẹn chu kỳ mùa thu-đông của Q3–Q4 năm 2026).

### 6.3. Hướng phát triển tiếp theo (Future Work)
- Bổ sung dữ liệu Q3 và Q4 năm 2026 khi FDA phát hành để kiểm chứng tính mùa vụ cả năm.
- Kết hợp thêm từ điển phân loại dược lý ATCvet hoặc dữ liệu nhãn thuốc (openFDA Animal & Veterinary Drug Labeling).

### 6.4. Phản ánh Cá nhân (Individual Reflections)
- **Đinh Công Khang (24120332):** *(Khó khăn gặp phải, cách vượt qua, bài học rút ra, cách sử dụng công cụ LLM)*
- **Trần Ngô Uyên Nhi (24120210):** *(Khó khăn gặp phải, cách vượt qua, bài học rút ra, cách sử dụng công cụ LLM)*
- **Huỳnh Tuấn Kiệt (24120356):** *(Khó khăn gặp phải, cách vượt qua, bài học rút ra, cách sử dụng công cụ LLM)*